# Inference Test — All Providers + VRAM Measurement

In [1]:
import subprocess, time
from pathlib import Path
from unified_local_llm_server import LocalLLMServer
from unified_local_llm_server.provider_registry import ProviderRegistry

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
registry = ProviderRegistry.load(ROOT / "providers.example.yaml")
server = LocalLLMServer(provider_registry=registry)

def vram_used_mb() -> int:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"]
    ).decode().strip()
    return int(out.splitlines()[0])

def vram_snapshot(label: str) -> int:
    mb = vram_used_mb()
    print(f"  VRAM [{label}]: {mb} MiB")
    return mb

In [2]:
# Unload all models before starting to ensure clean VRAM baseline
for _p in ["llama_cpp", "lm_studio", "ollama", "unsloth"]:
    try:
        srv = LocalLLMServer(provider_registry=registry)
        srv_inst = srv.provider_server(_p)
        srv.unload_all_models(_p)
    except Exception:
        pass
time.sleep(3)
print(f"VRAM baseline: {vram_used_mb()} MiB")

VRAM baseline: 18 MiB


## Provider Status

In [3]:
statuses = {}
for name in server.get_providers():
    statuses[name] = await server.check_provider_by_name(name)
    ok = statuses[name]["ok"]
    print(f"  {'✓' if ok else '✗'} {name:12} {statuses[name]['server_url']}")

  ✓ llama_cpp    http://127.0.0.1:9090
  ✓ lm_studio    http://127.0.0.1:1234
  ✓ ollama       http://127.0.0.1:11434
  ✓ unsloth      http://127.0.0.1:8899


## Config

In [4]:
PREFERRED_MODELS = {
    "ollama":    "gpt-oss:20b",
    "lm_studio": "openai/gpt-oss-20b",
    "unsloth":   "unsloth/gpt-oss-20b-GGUF@UD-Q4_K_XL",
    "llama_cpp": "gpt-oss-20b-MXFP4",
}

CTX_LENGTHS = [4096, 131072]
PROMPT = "Reply with one word: OK"
OPTIONS = {"num_predict": 4}

## VRAM Test — Each Provider × Each Context Length

In [5]:
vram_results = []

for provider in server.get_providers():
    if not statuses[provider]["ok"]:
        print(f"[{provider}] SKIP — not reachable")
        continue

    model = PREFERRED_MODELS.get(provider)
    if not model:
        try:
            model = await server.provider_server(provider).resolve_call_model(None)
        except Exception as exc:
            print(f"[{provider}] SKIP — {exc}")
            continue

    print(f"\n{'='*60}")
    print(f"  {provider} / {model}")
    print(f"{'='*60}")

    for ctx in CTX_LENGTHS:
        print(f"\n  -- context_length={ctx} --")

        # Unload first to get clean baseline
        try:
            server.unload_all_models(provider)
            time.sleep(2)
        except Exception:
            pass

        vram_before = vram_snapshot("before load")

        llm = server.load_model(provider, model, context_length=ctx)
        try:
            await llm.call(
                messages=[{"role": "user", "content": PROMPT}],
                options=OPTIONS,
            )
        except Exception as exc:
            print(f"  inference error: {exc}")

        time.sleep(2)
        vram_after = vram_snapshot("after load")
        delta = vram_after - vram_before
        print(f"  DELTA: {delta:+d} MiB")

        vram_results.append({
            "provider": provider,
            "model": model,
            "ctx": ctx,
            "vram_before": vram_before,
            "vram_after": vram_after,
            "delta": delta,
        })

    # Cleanup after provider
    try:
        server.unload_all_models(provider)
        time.sleep(2)
    except Exception:
        pass


  llama_cpp / gpt-oss-20b-MXFP4

  -- context_length=4096 --


  VRAM [before load]: 18 MiB
  inference error: llama_cpp /models/load failed (HTTP 500): 'Internal Server Error'


  VRAM [after load]: 18 MiB
  DELTA: +0 MiB

  -- context_length=131072 --


  VRAM [before load]: 18 MiB
  inference error: llama_cpp /models/load failed (HTTP 500): 'Internal Server Error'


  VRAM [after load]: 18 MiB
  DELTA: +0 MiB


...


  unsloth / unsloth/gpt-oss-20b-GGUF@UD-Q4_K_XL

  -- context_length=4096 --


  VRAM [before load]: 18 MiB


  VRAM [after load]: 11573 MiB
  DELTA: +11555 MiB

  -- context_length=131072 --


  VRAM [before load]: 18 MiB


  VRAM [after load]: 14563 MiB
  DELTA: +14545 MiB


## Summary

In [6]:
print(f"{'Provider':<12} {'ctx':>6}  {'before':>8}  {'after':>8}  {'delta':>8}")
print("-" * 55)
for r in vram_results:
    print(f"{r['provider']:<12} {r['ctx']:>6}  {r['vram_before']:>7} M  {r['vram_after']:>7} M  {r['delta']:>+7} M")

Provider        ctx    before     after     delta
-------------------------------------------------------
llama_cpp      4096       18 M       18 M       +0 M
llama_cpp    131072       18 M       18 M       +0 M
lm_studio      4096       18 M    11875 M   +11857 M
lm_studio    131072       18 M    18095 M   +18077 M
ollama         4096       18 M    12699 M   +12681 M
ollama       131072       18 M    16043 M   +16025 M
unsloth        4096       18 M    11573 M   +11555 M
unsloth      131072       18 M    14563 M   +14545 M


## Speed Test — Tokens per Second

In [7]:
SPEED_PROMPT = "Write a detailed explanation of how attention mechanisms work in transformer models."
SPEED_MAX_TOKENS = 150
SPEED_RUNS = 3
SPEED_CTXS = [500, 131072]

speed_results = []

for provider in server.get_providers():
    if not statuses[provider]["ok"]:
        print(provider, "SKIP — not reachable")
        continue

    model = PREFERRED_MODELS.get(provider)
    if not model:
        continue

    print()
    print("=" * 60)
    print("  " + provider + " / " + model)
    print("=" * 60)

    for ctx in SPEED_CTXS:
        print()
        print("  -- context_length=" + str(ctx) + " --")

        try:
            server.unload_all_models(provider)
            time.sleep(2)
        except Exception:
            pass

        try:
            llm = server.load_model(provider, model, context_length=ctx)
        except Exception as exc:
            print("  load error:", exc)
            continue

        run_times, run_tok = [], []

        for run in range(SPEED_RUNS):
            try:
                t0 = time.perf_counter()
                result = await llm.call(
                    messages=[{"role": "user", "content": SPEED_PROMPT}],
                    options={"num_predict": SPEED_MAX_TOKENS},
                )
                elapsed = time.perf_counter() - t0
                words = len(result.split())
                tok_est = words * 1.3
                tps = tok_est / elapsed if elapsed > 0 else 0
                run_times.append(elapsed)
                run_tok.append(tok_est)
                print("  run", run + 1, ":", round(elapsed, 1), "s  ~" + str(round(tps)) + " tok/s  (" + str(words) + " words)")
            except Exception as exc:
                print("  run", run + 1, "ERROR:", exc)

        if run_times:
            avg_s = sum(run_times) / len(run_times)
            avg_tps = sum(run_tok) / sum(run_times)
            print("  avg:", round(avg_s, 1), "s  ~" + str(round(avg_tps)) + " tok/s")
            speed_results.append({
                "provider": provider,
                "model": model,
                "ctx": ctx,
                "avg_s": round(avg_s, 1),
                "tok_per_s": round(avg_tps),
            })

    try:
        server.unload_all_models(provider)
        time.sleep(1)
    except Exception:
        pass



  llama_cpp / gpt-oss-20b-MXFP4

  -- context_length=500 --


  run 1 ERROR: llama_cpp /models/load failed (HTTP 500): 'Internal Server Error'
  run 2 ERROR: llama_cpp /models/load failed (HTTP 500): 'Internal Server Error'
  run 3 ERROR: llama_cpp /models/load failed (HTTP 500): 'Internal Server Error'

  -- context_length=131072 --


  run 1 ERROR: llama_cpp /models/load failed (HTTP 500): 'Internal Server Error'
  run 2 ERROR: llama_cpp /models/load failed (HTTP 500): 'Internal Server Error'
  run 3 ERROR: llama_cpp /models/load failed (HTTP 500): 'Internal Server Error'



  lm_studio / openai/gpt-oss-20b

  -- context_length=500 --


  run 1 : 3.7 s  ~32 tok/s  (92 words)


...

  run 2 : 1.0 s  ~122 tok/s  (95 words)


  run 3 : 1.5 s  ~83 tok/s  (98 words)
  avg: 2.4 s  ~53 tok/s

  -- context_length=131072 --


  run 1 : 4.4 s  ~30 tok/s  (102 words)


  run 2 : 1.0 s  ~135 tok/s  (103 words)


  run 3 : 1.0 s  ~138 tok/s  (109 words)
  avg: 2.1 s  ~64 tok/s


## Speed Summary

In [8]:
print("{:<12}  {:>8}  {:>7}  {:>7}".format("Provider", "ctx", "avg(s)", "tok/s"))
print("-" * 40)
for r in speed_results:
    print("{:<12}  {:>8}  {:>7.1f}  {:>7.0f}".format(r["provider"], r["ctx"], r["avg_s"], r["tok_per_s"]))


Provider           ctx   avg(s)    tok/s
----------------------------------------
lm_studio          500      1.9       64
lm_studio       131072      1.9       59
ollama             500      3.7       11
ollama          131072      3.0        9
unsloth            500      2.4       53
unsloth         131072      2.1       64


## Needle-in-a-Haystack — 110k Context

In [9]:
import random

NITH_CTX = 110000

NEEDLES = [
    {"depth": 0.10, "text": "The first secret code is: PINEAPPLE7742",  "key": "PINEAPPLE7742"},
    {"depth": 0.50, "text": "The second secret code is: MANGO3318",     "key": "MANGO3318"},
    {"depth": 0.75, "text": "The third secret code is: KIWI9051",       "key": "KIWI9051"},
]
NEEDLE_QUERY = (
    "There are exactly three secret codes hidden in the document above. "
    "List all three codes, one per line. Reply with ONLY the codes, nothing else."
)

_FILLER_SENTENCES = [
    "The annual rainfall in the central region averages about 482 millimeters per year.",
    "Municipal water treatment plants process approximately 35 million liters daily.",
    "Crop rotation schedules typically follow a three-year cycle of cereals and legumes.",
    "Regional transportation budgets are allocated based on population density metrics.",
    "Standardised testing protocols require calibration of instruments every 90 days.",
    "Warehouse inventory systems use barcode scanning for tracking inbound shipments.",
    "Public library cataloguing follows the Dewey Decimal Classification system.",
    "Meteorological stations record wind speed, humidity, and barometric pressure hourly.",
    "Urban planning guidelines recommend a minimum of 12 square meters of green space per resident.",
    "Quality assurance audits are conducted on a quarterly basis across all manufacturing lines.",
    "The historical archive contains over 1.2 million digitized documents from the 19th century.",
    "Soil composition analysis requires pH testing alongside nitrogen and phosphorus measurements.",
    "The regional power grid distributes energy from 14 substations across the district.",
    "Building codes mandate seismic resistance ratings for structures above three stories.",
    "Statistical sampling methods use a confidence interval of 95 percent for survey data.",
]


def build_filler(target_chars):
    parts = []
    total = 0
    while total < target_chars:
        s = random.choice(_FILLER_SENTENCES)
        parts.append(s)
        total += len(s) + 1
    return " ".join(parts)[:target_chars]


def build_haystack(target_tokens, chars_per_token=4.0):
    target_chars = int(target_tokens * chars_per_token)
    filler = build_filler(target_chars)
    for needle in sorted(NEEDLES, key=lambda n: n["depth"], reverse=True):
        insert_pos = int(len(filler) * needle["depth"])
        insert_pos = filler.rfind(". ", 0, insert_pos + 1)
        if insert_pos == -1:
            insert_pos = 0
        else:
            insert_pos += 2
        filler = filler[:insert_pos] + "\n\n" + needle["text"] + "\n\n" + filler[insert_pos:]
    return filler


def check_needles(response):
    normalized = response.upper().replace(" ", "")
    return {n["key"]: n["key"] in normalized for n in NEEDLES}


print("Building haystack for", NITH_CTX, "tokens (~" + str(NITH_CTX * 4 // 1000) + "k chars)...")
haystack = build_haystack(NITH_CTX)
print("Done:", len(haystack), "chars")


Building haystack for 110000 tokens (~440k chars)...
Done: 440121 chars


### Test — All Providers

In [10]:
nith_results = []

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant. Read the following document carefully, then answer the question.",
    },
    {
        "role": "user",
        "content": haystack + "\n\n" + NEEDLE_QUERY,
    },
]

for provider in server.get_providers():
    if not statuses[provider]["ok"]:
        print(provider, "SKIP")
        continue

    model = PREFERRED_MODELS.get(provider)
    if not model:
        continue

    print()
    print("=" * 60)
    print("  " + provider + " / " + model)
    print("=" * 60)

    try:
        server.unload_all_models(provider)
        time.sleep(2)
    except Exception:
        pass

    vram_before = vram_snapshot("before")

    try:
        llm = server.load_model(provider, model, context_length=NITH_CTX)
        t0 = time.perf_counter()
        result, usage = await llm.call(
            return_usage=True,
            messages=messages,
            temperature=0.0,
            options={"num_predict": 200},
        )
        elapsed = time.perf_counter() - t0
    except Exception as exc:
        print("  ERROR:", exc)
        nith_results.append({"provider": provider, "model": model, "error": str(exc)})
        continue

    vram_after = vram_snapshot("after")
    vram_delta = vram_after - vram_before

    prompt_tok = usage.get("prompt_tokens", 0)
    gen_tok = usage.get("completion_tokens", 0)
    gen_tps = gen_tok / elapsed if elapsed > 0 else 0
    prompt_tps = prompt_tok / elapsed if elapsed > 0 else 0

    found = check_needles(result)
    all_pass = all(found.values())
    status = "ALL PASS" if all_pass else ("PARTIAL" if any(found.values()) else "ALL FAIL")

    print("  Result       :", status)
    for key, ok in found.items():
        print("    " + key + ": " + ("PASS" if ok else "FAIL"))
    print("  Prompt tokens:", prompt_tok)
    print("  Gen tokens   :", gen_tok)
    print("  Gen speed    :", round(gen_tps, 1), "tok/s")
    print("  Total time   :", round(elapsed, 1), "s")
    print("  VRAM delta   :", vram_delta, "MiB")
    print("  Reply        :", result[:200])

    nith_results.append({
        "provider": provider,
        "model": model,
        "ctx": NITH_CTX,
        "found": found,
        "all_pass": all_pass,
        "status": status,
        "prompt_tokens": prompt_tok,
        "gen_tokens": gen_tok,
        "gen_tps": round(gen_tps, 1),
        "elapsed_s": round(elapsed, 1),
        "vram_before_mib": vram_before,
        "vram_after_mib": vram_after,
        "vram_delta_mib": vram_delta,
        "response": result,
    })

    try:
        server.unload_all_models(provider)
        time.sleep(1)
    except Exception:
        pass



  llama_cpp / gpt-oss-20b-MXFP4


  VRAM [before]: 18 MiB
  ERROR: llama_cpp /models/load failed (HTTP 500): 'Internal Server Error'

  lm_studio / openai/gpt-oss-20b


  VRAM [before]: 18 MiB


  VRAM [after]: 17015 MiB
  Result       : ALL PASS
    PINEAPPLE7742: PASS
    MANGO3318: PASS
    KIWI9051: PASS
  Prompt tokens: 72149
  Gen tokens   : 70
  Gen speed    : 1.7 tok/s
  Total time   : 41.3 s
  VRAM delta   : 16997 MiB
  Reply        : PINEAPPLE7742
MANGO3318
KIWI9051



  ollama / gpt-oss:20b


  VRAM [before]: 18 MiB


  VRAM [after]: 15531 MiB
  Result       : ALL PASS
    PINEAPPLE7742: PASS
    MANGO3318: PASS
    KIWI9051: PASS
  Prompt tokens: 72150
  Gen tokens   : 134
  Gen speed    : 4.2 tok/s
  Total time   : 31.8 s
  VRAM delta   : 15513 MiB
  Reply        : PINEAPPLE7742
MANGO3318
KIWI9051



  unsloth / unsloth/gpt-oss-20b-GGUF@UD-Q4_K_XL


  VRAM [before]: 18 MiB


  VRAM [after]: 14063 MiB
  Result       : ALL PASS
    PINEAPPLE7742: PASS
    MANGO3318: PASS
    KIWI9051: PASS
  Prompt tokens: 0
  Gen tokens   : 0
  Gen speed    : 0.0 tok/s
  Total time   : 29.8 s
  VRAM delta   : 14045 MiB
  Reply        : <think>We need to extract the three secret codes hidden in the document. The document includes lines: "The first secret code is: PINEAPPLE7742" then later "The second secret code is: MANGO3318" then l


### Results Summary

In [11]:
print("{:<12}  {:>8}  {:>12}  {:>10}  {:>8}  {:>8}  {:>14}  {:>14}  {:>14}".format(
    "Provider", "ctx", "Status", "Time(s)", "P.tok", "G.tok/s",
    "PINEAPPLE7742", "MANGO3318", "KIWI9051"))
print("-" * 110)
for r in nith_results:
    if "error" in r:
        print("{:<12}  {:>8}  ERROR: {}".format(r["provider"], NITH_CTX, r["error"][:50]))
        continue
    f = r["found"]
    print("{:<12}  {:>8}  {:>12}  {:>10.1f}  {:>8}  {:>8.1f}  {:>14}  {:>14}  {:>14}".format(
        r["provider"],
        r["ctx"],
        r["status"],
        r["elapsed_s"],
        r["prompt_tokens"],
        r["gen_tps"],
        "PASS" if f.get("PINEAPPLE7742") else "FAIL",
        "PASS" if f.get("MANGO3318") else "FAIL",
        "PASS" if f.get("KIWI9051") else "FAIL",
    ))


Provider           ctx        Status     Time(s)     P.tok   G.tok/s   PINEAPPLE7742       MANGO3318        KIWI9051
--------------------------------------------------------------------------------------------------------------
llama_cpp       110000  ERROR: llama_cpp /models/load failed (HTTP 500): 'Interna
lm_studio       110000      ALL PASS        41.3     72149       1.7            PASS            PASS            PASS
ollama          110000      ALL PASS        31.8     72150       4.2            PASS            PASS            PASS
unsloth         110000      ALL PASS        29.8         0       0.0            PASS            PASS            PASS
